In [5]:
%load_ext autoreload
%autoreload 2

In [6]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '../'))

In [7]:
import warnings

import json
from numpy import random

from model.optimization import load_model, eval_model
from model.dataset import get_dataframe
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split

DEFAULT_RANDOM_SEED = 774
random.mtrand._rand.seed(DEFAULT_RANDOM_SEED)
warnings.filterwarnings("ignore")

category="min_tpm_5"

In [3]:
test_data = get_dataframe(category=category, dataset="train")
subtypes = test_data["subtype"].unique()

NameError: name 'get_dataframe' is not defined

In [16]:
def run_tests(genes: int, report: bool = True):
  importances = json.loads(open(f"../../preprocessed/{category}/important_genes_random_forest_f1.json").readline())
  chosen_genes = list(set([y["gene"] for x in [sex_values[:genes] for subtype_items in importances.values() for sex_values in subtype_items.values()] for y in x]))
  print(f"Total chosen genes: {len(chosen_genes)}")

  train_data = get_dataframe(category=category, dataset="train")
  X, y = train_data[chosen_genes], train_data["subtype"]

  # X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.1, stratify=y)

  model = load_model(
    name="feature_selection_random_forest_f1",
    model_factory=lambda **kwargs: HistGradientBoostingClassifier(**kwargs, max_iter=1000, random_state=DEFAULT_RANDOM_SEED, class_weight="balanced"),
    dataset=(X, y)
  )
  
  # model = CalibratedClassifierCV(FrozenEstimator(model), method="sigmoid")
  # model.fit(X_valid, y_valid)

  return eval_model(model, dataset=(test_data[chosen_genes], test_data["subtype"]), report=report, use_threshold=False)
  

In [17]:
results = run_tests(genes=15)
print(results.report)

Total chosen genes: 252
   f1_weighted  f1_macro  recall_macro  balanced_accuracy  accuracy
0      0.86003  0.832078      0.826067           0.826067  0.860465
              precision    recall  f1-score   support

     BCRABL1       0.80      0.67      0.73        12
     DUX4IGH       1.00      1.00      1.00        21
   ETV6RUNX1       0.97      1.00      0.99        33
       HYPER       0.68      0.83      0.75        23
        HYPO       0.88      0.64      0.74        11
       KMT2A       1.00      1.00      1.00        19
        PAX5       0.83      0.77      0.80        13
      PHlike       0.73      0.70      0.71        23
    TCF3PBX1       1.00      1.00      1.00        11
      iAMP21       0.57      0.67      0.62         6

    accuracy                           0.86       172
   macro avg       0.85      0.83      0.83       172
weighted avg       0.87      0.86      0.86       172

